# Tutorial 5: Fine-Tuning

By the end you will be able to:

- decide whether a problem calls for fine-tuning, or for prompting or retrieval instead;
- prepare a dataset: tokenize sentence pairs once, and pad each batch dynamically;
- fine-tune a classifier with **`Trainer`** and evaluate it with **`evaluate`**;
- write the same training loop by hand, and say what `Trainer` was doing for you;
- format conversations with a **chat template**;
- fine-tune a language model on conversations with TRL's **`SFTTrainer`**;
- train a **LoRA** adapter instead of the full model, then merge it back.

**Hardware.** Use a GPU (Colab: *Runtime → Change runtime type → T4 GPU*). Every cell also runs on a CPU, but
the training cells take minutes on a GPU and much longer without one.

## 0. Mental model: when and how to fine-tune

<img src="images/finetuning_cards/When_and_how_to_finetune.png" alt="When and how to fine-tune card: prompting guides an existing skill, RAG retrieves relevant knowledge, fine-tuning learns task behaviour; BERT gets a new 2-class head trained with class cross-entropy, SmolLM2 keeps its LM head and is trained with next-token cross-entropy. Same learning loop, different targets." width="650">

Fine-tuning is the expensive way to change a model's behaviour. Check the cheaper ones first.

| Approach | What changes | Costs | Right when |
|---|---|---|---|
| **Prompting** | Nothing. Instructions only. | Tokens per call | The model can already do it, given clear instructions |
| **RAG** | Nothing. Context is retrieved. | Tokens + an index | It lacks *knowledge*, and that knowledge changes |
| **Fine-tuning** | The weights | GPU time + data + a model to host | It lacks a *skill*, a format, or a domain style |

Under the hood it is ordinary supervised learning, starting from pretrained weights instead of random ones:

```text
batch → forward pass → loss vs labels → backward pass → optimizer step → repeat
```

The notebook does this twice:

| | Part 1 | Part 2 |
|---|---|---|
| Model | `bert-base-uncased` (encoder) | `SmolLM2-135M` (decoder) |
| Task | Are two sentences paraphrases? | Hold a conversation |
| New layer | A random classification head | None: the LM head already predicts tokens |
| Loss | Cross-entropy over 2 classes | Cross-entropy over the next token |
| Trainer | `Trainer` (Transformers) | `SFTTrainer` (TRL), which subclasses `Trainer` |

## 1. Install the libraries

<img src="images/finetuning_cards/Install_libraries.png" alt="Fine-tuning toolbox card: transformers for models and Trainer, datasets to load and map, evaluate for metrics, accelerate for devices, trl for SFTTrainer, peft for LoRA and scikit-learn for F1, installed with one pip command." width="650">

In [ ]:
!pip -q install -U transformers datasets evaluate accelerate peft trl scikit-learn

### Code walkthrough: what each library adds

- **`datasets`**, **`evaluate`**: loading data from the Hub, and standard metrics (tutorial 0).
- **`accelerate`**: `Trainer` uses it to place tensors on whatever hardware exists. It refuses to start
  without it.
- **`trl`**: Transformer Reinforcement Learning. Despite the name, it is where `SFTTrainer` lives (Part 2).
- **`peft`**: Parameter-Efficient Fine-Tuning, which provides LoRA (section 10).
- **`scikit-learn`**: needed by the GLUE metric for F1.

# Part 1: Fine-tuning a classifier

## 2. The dataset: MRPC

<img src="images/finetuning_cards/MRPC_dataset.png" alt="MRPC card: pairs of sentences labelled 1 equivalent or 0 not_equivalent, loaded with load_dataset(&#x27;nyu-mll/glue&#x27;, &#x27;mrpc&#x27;); 3,668 training pairs, a validation split for measuring and a test split with hidden labels (-1)." width="650">

**MRPC** (Microsoft Research Paraphrase Corpus) is one of the ten tasks in the GLUE benchmark. Each example
is a pair of sentences and a label saying whether they mean the same thing. It is small (3,668 training
pairs), so a full fine-tune takes a few minutes.

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
print(raw_datasets)
print()
print(raw_datasets["train"][0])
print()
print(raw_datasets["train"].features["label"])

### Code walkthrough: a `DatasetDict` with three splits

- **`load_dataset("nyu-mll/glue", "mrpc")`**: the second argument picks one task out of GLUE. The course
  writes `"glue"`; current `huggingface_hub` versions want the full `namespace/name` id.
- **Three splits.** `train` to learn from, `validation` to measure while developing, `test` for a final
  number. GLUE's test labels are hidden (all `-1`), so we evaluate on `validation`.
- **Labels are already integers.** The `ClassLabel` feature maps `0 → not_equivalent`, `1 → equivalent`.
- The data is stored on disk in Apache Arrow format and loaded lazily, so large datasets do not need to fit
  in memory.

## 3. Tokenizing sentence pairs

<img src="images/finetuning_cards/BERT_sentence_pair_tokens.png" alt="Sentence pair card: tokenizer(sentence1, sentence2) builds [CLS] A [SEP] B [SEP], with token_type_ids 0 for sentence A and 1 for sentence B, plus input_ids and attention_mask." width="650">

The model needs `input_ids`, not strings. For a pair task, the tokenizer takes **both** sentences and joins
them the way BERT saw pairs during pretraining.

In [ ]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

inputs = tokenizer("This is the first sentence.", "This is the second one.")
print(inputs)
print()
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"]))

### Code walkthrough: `[CLS] A [SEP] B [SEP]` and `token_type_ids`

- **Two arguments, one sequence.** Passing two strings produces `[CLS] sentence1 [SEP] sentence2 [SEP]`.
  Do not concatenate the strings yourself: the tokenizer knows the format its model expects.
- **`token_type_ids`** mark which sentence each token belongs to: `0` for the first, `1` for the second.
  BERT was pretrained with them (next-sentence prediction), so it uses them. Models pretrained without
  them (DistilBERT, RoBERTa) do not return them. This is one more reason the tokenizer and model must come
  from the same checkpoint.

## 4. Tokenize the dataset, pad per batch

<img src="images/finetuning_cards/Tokenize_dataset_pad_per_batch.png" alt="Tokenize once, pad each batch card: map(tokenize_function, batched=True) adds input_ids, token_type_ids and attention_mask columns of different lengths; DataCollatorWithPadding pads each batch only to its own longest sequence." width="650">

Tokenize everything once, up front. Do **not** pad yet: padding happens later, per batch.

In [ ]:
from transformers import DataCollatorWithPadding

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_datasets)

# dynamic padding on 8 examples
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
print("\nlengths before padding:", [len(x) for x in samples["input_ids"]])

batch = data_collator(samples)
print("batch shapes:", {k: tuple(v.shape) for k, v in batch.items()})

### Code walkthrough: `map(batched=True)` and dynamic padding

- **`map(tokenize_function, batched=True)`** calls the function on chunks of rows, so the fast (Rust)
  tokenizer processes many texts per call. The result is cached on disk; re-running the cell is instant.
- **`map` adds columns.** The new `input_ids`, `token_type_ids` and `attention_mask` sit beside the original
  text columns.
- **`truncation=True`** caps pairs at the model's maximum length (512 for BERT).
- **No padding in `tokenize_function`.** Padding the whole dataset to its longest example would waste
  compute on every batch.
- **`DataCollatorWithPadding`** pads each batch to the longest sequence *in that batch* (here 67 tokens,
  not 512) and builds the `attention_mask`. The text columns must be dropped first: tensors cannot hold
  strings.
- **TPUs are the exception.** They prefer fixed shapes, so pad everything to one length there.

## 5. Fine-tuning with the `Trainer` API

<img src="images/finetuning_cards/Trainer_API.png" alt="Trainer card: BERT with a random classifier head, TrainingArguments (learning_rate 2e-5, 3 epochs, batch 16, eval_strategy epoch) and compute_metrics go into Trainer, which trains on MRPC train and evaluates on validation each epoch; logits go through argmax to accuracy and F1." width="650">

Three pieces: a model, a `compute_metrics` function, and a `TrainingArguments` object.

In [ ]:
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification

labels = raw_datasets["train"].features["label"].names
id2label = dict(enumerate(labels))
label2id = {name: i for i, name in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=2, id2label=id2label, label2id=label2id
)

def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

### Code walkthrough: a body you keep, a head you train

- **The load report is expected.** It lists the `cls.*` weights as `UNEXPECTED` (BERT's pretraining heads,
  dropped) and `classifier.weight` / `classifier.bias` as `MISSING` (a new 2-class head, created with
  **random weights**). Before training, its predictions are noise. Be alarmed only when weights you expected
  to be pretrained show up as `MISSING`.
- **`id2label` / `label2id`** come from the dataset's `ClassLabel`, not hard-coded. They are saved in the
  model config, so a pipeline later prints `equivalent` instead of `LABEL_1`.
- **`compute_metrics`** receives `(logits, labels)` as NumPy arrays. `np.argmax` picks the winning class
  (no softmax needed: it cannot change the winner). `evaluate.load("glue", "mrpc")` returns accuracy **and**
  F1, the two numbers the GLUE benchmark reports for this task.

In [ ]:
import torch
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    "bert-mrpc",
    eval_strategy="epoch",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

### Code walkthrough: `TrainingArguments` and `Trainer`

- **`"bert-mrpc"`** is the only required argument: the output directory for checkpoints. Every other
  argument has a working default.
- **`eval_strategy="epoch"`** runs `compute_metrics` on the validation set after each epoch. Without it,
  training reports only the loss, which is hard to interpret.
- **`learning_rate=2e-5`** is the standard band for BERT fine-tuning (`2e-5` to `5e-5`). Pretrained weights
  are already good; large steps destroy them.
- **`fp16=True`** (mixed precision) roughly halves memory and speeds up training on a GPU. It is switched off
  on a CPU, where it is not supported.
- **`report_to="none"`** stops `Trainer` logging to Weights & Biases or other trackers.
- **`processing_class=tokenizer`** replaces the older `tokenizer=` argument. It is saved with every
  checkpoint, so a checkpoint is never separated from its tokenizer.
- **The course reports about 86% accuracy and 90% F1** after 3 epochs. Exact numbers vary run to run,
  because the head starts from different random weights.
- **To start over, recreate the model.** Calling `train()` again continues from the current weights.

Three more arguments from the course's *advanced features* list:

- **`gradient_accumulation_steps=4`** with `per_device_train_batch_size=4` gives an effective batch of 16
  in a quarter of the memory. It is the first fix for an out-of-memory error.
- **`lr_scheduler_type="cosine"`** changes the learning-rate schedule. The default is a linear decay to 0.
- **`bf16=True`** instead of `fp16` on newer GPUs (Ampere and later). The Colab T4 supports only `fp16`.

In [ ]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

preds = np.argmax(predictions.predictions, axis=-1)
metric = evaluate.load("glue", "mrpc")
print(metric.compute(predictions=preds, references=predictions.label_ids))

### Code walkthrough: `predict()` returns logits

- **`trainer.predict`** returns a named tuple: `predictions` (logits, shape `[408, 2]`), `label_ids`
  (`[408]`) and `metrics`.
- **Logits, not labels.** One score per class per example. `argmax` along the last axis turns them into
  class ids, exactly what `compute_metrics` did during training.

## 6. The same training, written by hand

<img src="images/finetuning_cards/Manual_training_loop.png" alt="Trainer&#x27;s engine card: prepare the dataset (drop text columns, rename label to labels, set_format torch), build a DataLoader, then loop: move batch to device, forward, loss.backward(), optimizer.step(), lr_scheduler.step(), optimizer.zero_grad()." width="650">

`Trainer` hides a loop of about twenty lines. Writing it once shows what it does, and it is what you need
when you want to change a step that `Trainer` does not expose.

In [ ]:
from torch.utils.data import DataLoader

tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
print(tokenized_datasets["train"].column_names)

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

### Code walkthrough: the three things `Trainer` did silently

- **`remove_columns`**: drop every column the model's `forward()` does not accept. `Trainer` does this
  automatically.
- **`rename_column("label", "labels")`**: the model's argument is called `labels`. When it receives them,
  it computes the loss itself.
- **`set_format("torch")`**: return PyTorch tensors instead of Python lists.
- **`shuffle=True`** for training only. The collator still pads each batch dynamically.

In [ ]:
from torch.optim import AdamW
from transformers import get_scheduler

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=2, id2label=id2label, label2id=label2id
)
optimizer = AdamW(model.parameters(), lr=5e-5)

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
print(device, "|", num_training_steps, "training steps")

In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

In [ ]:
metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    predictions = torch.argmax(outputs.logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

print(metric.compute())

### Code walkthrough: one training step

<img src="images/finetuning_cards/Manual_evaluation.png" alt="Evaluation card: model.eval() disables dropout and torch.no_grad() stops gradient tracking; validation batches go through the model, argmax turns logits into labels, metric.add_batch collects them and metric.compute() gives the score, with no optimizer step." width="650">

- **A fresh model.** The first one has already been trained in section 5; continuing from it would not be a
  fair comparison.
- **`AdamW`** is `Trainer`'s default optimizer: Adam with weight decay applied separately from the gradient
  update.
- **`get_scheduler("linear", ...)`** decays the learning rate from `5e-5` to `0` over all steps
  (3 epochs × 459 batches = 1,377 steps).
- **The step, in order:** forward pass (the model returns `loss` because `labels` are in the batch),
  `loss.backward()` computes gradients, `optimizer.step()` updates the weights, `lr_scheduler.step()` lowers
  the learning rate, `optimizer.zero_grad()` clears gradients for the next batch. Forget `zero_grad()` and
  gradients accumulate across batches.
- **Evaluation:** `model.eval()` turns off dropout, `torch.no_grad()` skips gradient tracking (tutorial 2,
  section 10). `metric.add_batch` accumulates predictions; `metric.compute()` gives the final scores.

**Running on several GPUs.** The course's next step is 🤗 Accelerate: create `accelerator = Accelerator()`,
pass the dataloaders, model and optimizer through `accelerator.prepare(...)`, delete the `.to(device)`
lines, and replace `loss.backward()` with `accelerator.backward(loss)`. The same script then runs on one
GPU, several GPUs or a TPU.

## 7. Save, reload and share

<img src="images/finetuning_cards/Save_reload_share.png" alt="Save card: trainer.save_model writes weights, config with id2label and the tokenizer to bert-mrpc/final; pipeline(&#x27;text-classification&#x27;) takes a text and text_pair and returns equivalent or not_equivalent; push_to_hub shares it after login." width="650">

Save the model with its tokenizer, then load it back through `pipeline()`.

In [ ]:
from transformers import pipeline

SAVE_DIR = "bert-mrpc/final"
trainer.save_model(SAVE_DIR)

classifier = pipeline("text-classification", model=SAVE_DIR)

pairs = [
    ("The company posted record profits this quarter.",
     "This quarter the firm reported its highest earnings ever."),
    ("The company posted record profits this quarter.",
     "The company's CEO announced her retirement."),
]
for a, b in pairs:
    out = classifier({"text": a, "text_pair": b})
    print(f"{out['label']:>15}  {out['score']:.3f}  {b}")

### Code walkthrough: what gets saved

- **`trainer.save_model`** writes the weights, the config (with `id2label`) and, because it was passed as
  `processing_class`, the tokenizer. That is why the pipeline prints `equivalent` / `not_equivalent`.
- **`{"text": a, "text_pair": b}`** passes a sentence pair, tokenized the same way as in training.
- **Sharing.** Run `huggingface_hub.login()`, then `trainer.push_to_hub()` uploads the model, tokenizer
  and an auto-generated model card. Anyone can then load it by name with `from_pretrained`.

# Part 2: Fine-tuning a chat model

Part 1 trained a new head for a closed task. Most LLM fine-tuning instead trains the model to **respond**:
follow instructions, hold a conversation, use a format. This is **supervised fine-tuning (SFT)**, and it is
how a base model becomes an instruct model.

## 8. Chat templates

<img src="images/finetuning_cards/Chat_templates.png" alt="Chat templates card: apply_chat_template turns system, user and assistant messages into SmolLM2&#x27;s ChatML text with &lt;|im_start|&gt; and &lt;|im_end|&gt;; training data already contains replies, while add_generation_prompt=True cues the next reply at inference. Train and infer with the same template." width="650">

A conversation is a list of messages. A **chat template** turns that list into the exact string, with the
exact special tokens, the model was trained on.

In [ ]:
from transformers import AutoTokenizer

chat_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi! How can I help?"},
]

print(chat_tokenizer.apply_chat_template(messages, tokenize=False))
print("-" * 40)
print(chat_tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True))

### Code walkthrough: roles, special tokens and the generation prompt

- **Roles.** `system` sets standing behaviour, `user` is the person, `assistant` is the model.
- **The template is model-specific.** SmolLM2 uses the ChatML format (`<|im_start|>role ... <|im_end|>`);
  Llama, Mistral and Gemma each use their own. A model only follows a conversation formatted in *its*
  template, which is why the template ships inside the tokenizer.
- **`add_generation_prompt=True`** appends `<|im_start|>assistant` at the end: the cue for the model to
  start its reply. Use it at inference time, never in training data, which already contains the reply.
- **Base models have no template.** They were trained on plain text. Teaching a base model a template is
  the first job of SFT, which is the next section.

## 9. Supervised fine-tuning with TRL

<img src="images/finetuning_cards/Supervised_finetuning_TRL.png" alt="SFT card: the SmolLM2-135M base model only continues text; SFTTrainer trains it on smoltalk everyday-conversations with the SmolLM2-Instruct template (max_steps 300, batch 8, learning rate 5e-5, max_length 512) using next-token loss, so it learns to reply as the assistant and end with &lt;|im_end|&gt;." width="650">

We turn **`SmolLM2-135M`**, a *base* model that only continues text, into a model that answers in turns,
using `HuggingFaceTB/smoltalk`, the dataset SmolLM2-Instruct was trained on. We use its small
`everyday-conversations` subset (2,260 conversations).

First, look at how the base model behaves before training.

In [ ]:
from transformers import AutoModelForCausalLM

BASE = "HuggingFaceTB/SmolLM2-135M"
base_tokenizer = AutoTokenizer.from_pretrained(BASE)
base_model = AutoModelForCausalLM.from_pretrained(BASE)

prompt = "What should I cook for dinner tonight?"
ids = base_tokenizer(prompt, return_tensors="pt")
out = base_model.generate(**ids, max_new_tokens=40, do_sample=False,
                          pad_token_id=base_tokenizer.eos_token_id)
print(base_tokenizer.decode(out[0]))

### Before you run the training cell

The base model does not *answer*. It continues the text the way web pages do: here it invents a `A:` line,
then repeats `I'm not sure.` until it runs out of tokens. It has never seen a conversation format, so it
has no notion of whose turn it is, or of when a reply should end.

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

dataset = load_dataset("HuggingFaceTB/smoltalk", "everyday-conversations")
print(dataset)
print(dataset["train"][0]["messages"][:2])

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

sft_args = SFTConfig(
    output_dir="smollm2-135m-sft",
    chat_template_path="HuggingFaceTB/SmolLM2-135M-Instruct",
    eos_token="<|im_end|>",
    max_steps=300,
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    max_length=512,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=BASE,
    args=sft_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)
sft_trainer.train()

### Code walkthrough: `SFTConfig` and `SFTTrainer`

- **The dataset has a `messages` column**, so TRL recognises it as *conversational* and applies the chat
  template to every example. No `tokenize_function`, no collator: `SFTTrainer` handles both.
- **`chat_template_path`** gives the base model the template of the instruct model. **`eos_token`**
  makes `<|im_end|>` the end-of-turn token, so the trained model learns to *stop* after its reply.
- **`model=BASE`** is just a name: `SFTTrainer` loads the model and tokenizer itself.
- **The loss** is next-token cross-entropy (tutorial 2), averaged over the tokens of each conversation.
  Set `assistant_only_loss=True` to train on the assistant's replies only.
- **`max_steps=300`** caps the run (a few minutes on a T4). The course uses 1,000 steps on the full
  `smoltalk`; raise it once the pipeline works.
- **`learning_rate=5e-5`**, `max_length=512` (longer conversations are truncated), and evaluation every 100
  steps.
- **Precision.** `SFTConfig` defaults to `bf16=True`, which fails on GPUs without bfloat16 (the Colab T4).
  `use_bf16` picks `bf16` where it is supported and falls back to `fp16`.
- **Different defaults from `TrainingArguments`.** `SFTConfig` turns on `gradient_checkpointing` (saves
  memory, costs about 20% speed) and logs every 10 steps by default.

In [ ]:
chat = pipeline("text-generation", model=sft_trainer.model, tokenizer=sft_trainer.processing_class)

question = [{"role": "user", "content": "What should I cook for dinner tonight?"}]
reply = chat(question, max_new_tokens=80, do_sample=False)
print(reply[0]["generated_text"][-1]["content"])

### Code walkthrough: chatting with the result

- **A list of messages in, a list of messages out.** The pipeline applies the chat template (with the
  generation prompt), generates, and returns the conversation with the new assistant turn appended.
  `[-1]["content"]` is the reply.
- **Compare with the base model's output.** The reply should now be an *answer* in the assistant's turn,
  ending on its own: that is what the template and `eos_token` teach. If it still rambles, train longer
  (raise `max_steps`).
- **Judge it honestly.** 135M parameters and 300 steps teach the *format* of a conversation, not much
  knowledge. Loss alone hides this, so the course stresses reading real outputs.

## 10. LoRA: training an adapter instead of the model

<img src="images/finetuning_cards/LoRA_adapter_training.png" alt="LoRA card: the pretrained weight W stays frozen while a trainable low-rank adapter B times A, scaled by alpha over r, is added: y = Wx + (alpha/r) BAx; config r=8, lora_alpha=16, lora_dropout=0.05, target_modules all-linear, learning rate 1e-4." width="650">

Full fine-tuning updates every weight and stores gradients and optimizer state for each, roughly 12-16 bytes
per parameter. That is fine for 135M parameters and impossible for 7B on one consumer GPU.

**LoRA** (Low-Rank Adaptation) freezes the pretrained weights and trains two thin matrices beside selected
layers: instead of a full update `ΔW`, it learns `ΔW = B·A`, where `A` and `B` have rank `r`, far below the
layer's width.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

lora_args = SFTConfig(
    output_dir="smollm2-135m-lora",
    chat_template_path="HuggingFaceTB/SmolLM2-135M-Instruct",
    eos_token="<|im_end|>",
    max_steps=300,
    per_device_train_batch_size=8,
    learning_rate=1e-4,
    max_length=512,
    logging_steps=25,
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    report_to="none",
)

lora_trainer = SFTTrainer(
    model=BASE,
    args=lora_args,
    train_dataset=dataset["train"],
    peft_config=peft_config,
)
lora_trainer.model.print_trainable_parameters()
lora_trainer.train()

### Code walkthrough: the LoRA knobs

- **`r=8`** is the rank, the main capacity knob. The course recommends 4-32: start at 4-8 and raise it only
  if the model underfits.
- **`lora_alpha=16`** scales the adapter's contribution by `alpha / r`. The convention is `alpha = 2r`.
- **`lora_dropout=0.05`** (0.05-0.1) regularises the adapter; **`bias="none"`** leaves biases frozen.
- **`target_modules="all-linear"`** puts adapters on every linear layer. Naming a few (for example
  `["q_proj", "v_proj"]`) uses less memory and adapts less.
- **`peft_config=`** is the only change to the trainer. `SFTTrainer` wraps the model with PEFT itself.
- **`print_trainable_parameters()`** shows the payoff: 2.4M of 137M weights, about 1.8%, train here.
- **`learning_rate=1e-4`**: adapters start from zero, so they tolerate (and need) a higher rate than full
  fine-tuning. TRL's docs recommend about `1e-4`.
- **QLoRA.** Loading the frozen base in 4-bit (pass `quantization_config=BitsAndBytesConfig(load_in_4bit=True)`
  to `SFTTrainer`) is what fits a 7B model on a 16 GB GPU.

In [ ]:
import os

ADAPTER_DIR = "smollm2-135m-lora/adapter"
lora_trainer.save_model(ADAPTER_DIR)
size_mb = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f)) for f in os.listdir(ADAPTER_DIR)) / 1e6
print(f"adapter on disk: {size_mb:.1f} MB")

merged = lora_trainer.model.merge_and_unload()
merged.save_pretrained("smollm2-135m-lora/merged")
lora_trainer.processing_class.save_pretrained("smollm2-135m-lora/merged")

chat = pipeline("text-generation", model="smollm2-135m-lora/merged")
reply = chat(question, max_new_tokens=80, do_sample=False)
print(reply[0]["generated_text"][-1]["content"])

### Code walkthrough: adapters, merging and reloading

<img src="images/finetuning_cards/Save_and_merge_adapters.png" alt="Adapters card: keep the adapter (save_model writes adapter weights and config, PeftModel.from_pretrained puts it back on the base) or merge it (merge_and_unload folds alpha/r BA into W, giving an ordinary model saved with its tokenizer and chat template)." width="650">

- **Saving a PEFT model saves only the adapter**: about 13 MB here, against about 550 MB for the full model
  in fp32. You can keep many task-specific adapters for one base model.
- **Loading an adapter later:** load the base with `AutoModelForCausalLM.from_pretrained(BASE)`, then
  `PeftModel.from_pretrained(base, ADAPTER_DIR)`.
- **`merge_and_unload()`** adds `B·A` into the frozen weights and removes the adapter layers. The result is
  an ordinary model: no PEFT needed to load it, and no extra inference cost.
- **Save the tokenizer with the merged model**: it carries the chat template the adapter was trained with.

## 11. Reading a training run

<img src="images/finetuning_cards/Read_a_training_run.png" alt="Reading a run card: loss down and metric up is healthy; training loss down and validation loss up is overfitting; loss that barely moves means check learning rate, labels and a decoded batch; a suspiciously tiny loss means memorisation or leakage. Then check real replies for role, usefulness and stopping." width="650">

The numbers `Trainer` and `SFTTrainer` print are enough to diagnose most problems.

- **Training loss falls, eval metric improves.** Working as intended.
- **Training loss falls, eval loss rises (or eval accuracy stalls).** Overfitting: fewer epochs, more data, or
  a lower learning rate.
- **Loss barely moves.** Something is broken: a learning rate far too low, labels misaligned with inputs,
  or text truncated to nothing. Decode one batch before touching hyperparameters:
  `tokenizer.decode(next(iter(train_dataloader))["input_ids"][0])`.
- **Loss suspiciously close to 0.** Memorisation, or the labels leaked into the inputs.
- **SFT only:** `mean_token_accuracy` in the SFT log is the share of tokens the model predicts exactly. And
  read real outputs: a low loss does not guarantee consistent formatting or a model that stops.

**Pitfalls.**

- **Evaluating on training data.** Keep a held-out split untouched.
- **A learning rate from another setting.** About `2e-5` for full fine-tuning, about `1e-4` for LoRA. `1e-3`
  destroys pretrained weights.
- **The wrong chat template.** Train and infer with the same one.
- **Catastrophic forgetting.** Fine-tuning hard on a narrow task degrades general ability. LoRA adapters,
  which can be detached, limit the damage.